## CSV Ingestion - Load Hackerrank SQL Practice Data


### Instructions
1. Load data to each catalog volume: Use this template to query tables in Hackerrank and paste them into CSV files using notepad or other text editor.

<br>
<pre><code><b>SELECT</b> 'ID,NAME,COUNTRYCODE,DISTRICT,POPULATION' AS csv_row
<b>UNION ALL</b>
<b>SELECT</b> CONCAT(ID, ',', NAME, ',', COUNTRYCODE, ',', DISTRICT, ',', POPULATION)
<b>FROM</b> CITY;
</code></pre>
<br>
2. Excecute each cell of this notebook.
3. Check that Delta tables are successfuly created.
4. Go ahead and start working on hr_practice.

In [0]:
-- Checks for volume path and files
LIST "/Volumes/dev_world/bronze/raw_hackerrank/cities"

In [0]:
-- Checks for files content (Esto es solo consultar los datos del volumen sin crear nada en Databricks)
SELECT * FROM csv.`/Volumes/dev_world/bronze/raw_hackerrank/cities/*.csv`;

In [0]:
-- Creates CITY table with additional _metadata (Acá sí creamos la tabla delta)
CREATE OR REPLACE TABLE dev_world.bronze.city AS
SELECT 
    cast(id as INT) as id,
    cast(name as STRING) as name,
    cast(countrycode as STRING) as countrycode,
    cast(district as STRING) as district,
    cast(population as INT) as population,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/bronze/raw_hackerrank/cities/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.bronze.city LIMIT 10;

In [0]:
-- Let's create a silver Squema for Cities
CREATE SCHEMA IF NOT EXISTS dev_world.silver;

-- And a volume inside that layer
CREATE VOLUME IF NOT EXISTS dev_world.silver.city;

In [0]:
-- Creates a siver CITY table with additional _metadata (Silver no debería ingestar datos pero bueno, se entiende que esos datos evolucionaron y por eso están en esta instancia :) )
CREATE OR REPLACE TABLE dev_world.silver.city AS
SELECT 
    cast(id as INT) as id,
    cast(name as STRING) as name,
    cast(countrycode as STRING) as countrycode,
    cast(district as STRING) as district,
    cast(population as INT) as population,
    current_timestamp() AS _ingestion_time,
    _metadata.file_name AS _source_file
FROM read_files(
    '/Volumes/dev_world/silver/city/',
    format => 'csv',
    header => true
);

-- Consulta los primeros 10 registros
SELECT * FROM dev_world.silver.city LIMIT 10;